# 10b — Part Assembly with Transforms

**Curriculum slot:** Tier 3, slot 10.5.
**Prerequisite:** 10 — Parts Basics.

## Purpose

Slot 10 showed how to *tag* existing geometry with ``g.parts.register``.
That's the simplest part workflow but doesn't demonstrate the most
useful application of the Parts composite: **instancing a template
geometry multiple times with transforms.**

The canonical apeGmsh idiom:

```python
col = Part(name="column").begin()
# ... build the column geometry inside its own session ...
col.end()

with apeGmsh(model_name="assembly") as g:
    g.parts.add(col, label="col_0", translate=(0.0, 0, 0))
    g.parts.add(col, label="col_1", translate=(4.0, 0, 0))
    g.parts.add(col, label="col_2", translate=(8.0, 0, 0))
```

Under the hood ``Part.end()`` auto-persists the Part's geometry
to a STEP file (OS tempfile, garbage-collected with the Part),
and each ``g.parts.add(col, translate=...)`` imports that STEP
into the main session with the transform applied. This is the
same mechanism ``g.parts.import_step(file_path, translate=...)``
uses directly when the part comes from a pre-built CAD file —
ideal for dropping a library of CAD parts into a scene.

## Problem — three identical columns in a row

A 3 m vertical cantilever beam template, replicated at $x = 0$,
$x = 4$, $x = 8$. Each column gets its own fixed base and tip
point load $P$. Each column's tip deflection must match the
classical $P L^{3} / (3\,E\,I)$ — the transform (pure
translation) doesn't alter stiffness.


## 1. Imports and parameters


In [1]:
import openseespy.opensees as ops

from apeGmsh import apeGmsh
from apeGmsh.core.Part import Part

# --- Geometry ---
L = 3.0                     # column height
N_COLS = 3                  # how many instances to place
DX = 4.0                    # spacing between columns in +x
COLUMN_X_OFFSETS = [i * DX for i in range(N_COLS)]

# --- Loading ---
P = 10_000.0                # tip horizontal load (pushed in +x) [N]

# --- Material ---
E  = 2.1e11
nu = 0.3
G  = E / (2 * (1 + nu))
A, Iy, Iz, J = 1e-3, 1e-5, 1e-5, 2e-5

LC = L / 10.0


## 2. Build the template Part in isolation

``Part(name=...).begin()`` opens an isolated Gmsh session for the
part. We build exactly the geometry we want (line + two
endpoints). ``end()`` auto-persists the Part to a tempfile.


In [2]:
col_template = Part(name="cantilever_column").begin(verbose=False)
col_template.model.geometry.add_point(0.0, 0.0, 0.0, lc=LC)
col_template.model.geometry.add_point(0.0, 0.0, L,   lc=LC)
# Note: the line comes along automatically when add_line is called.
# Using the geometry add_line keeps the part's internal tag space clean.
p0 = 1   # first point added inside the part; its OCC tag is 1
p1 = 2   # second point; OCC tag 2
col_template.model.geometry.add_line(p0, p1)
col_template.model.sync()
col_template.end()


## 3. Assemble three instances in the main session

Each call to ``g.parts.add(part, label=..., translate=...)``
imports the template's STEP into the assembly session at the
given translate offset and registers it as a new ``Instance``.


In [3]:
g_ctx = apeGmsh(model_name="10b_part_assembly", verbose=False)
g = g_ctx.__enter__()

for i, x_off in enumerate(COLUMN_X_OFFSETS):
    g.parts.add(
        col_template,
        label=f"col_{i}",
        translate=(x_off, 0.0, 0.0),
    )

print(f"parts registered: {g.parts.labels()}")
for lbl in g.parts.labels():
    print(f"  {lbl} entities: {dict(g.parts.get(lbl).entities)}")


parts registered: ['col_0', 'col_1', 'col_2']
  col_0 entities: {1: [1]}
  col_1 entities: {1: [2]}
  col_2 entities: {1: [3]}


## 4. Mesh


In [4]:
g.mesh.generation.generate(1)
fem = g.mesh.queries.get_fem_data()
print(f"mesh: {fem.info.n_nodes} nodes, {fem.info.n_elems} elements")

for lbl in sorted(g.parts.labels()):
    n = len(list(fem.nodes.get(target=lbl).ids))
    e = sum(len(gr.ids) for gr in fem.elements.get(target=lbl))
    print(f"  {lbl}: {n} nodes, {e} elements")


mesh: 9 nodes, 12 elements
  col_0: 3 nodes, 2 elements
  col_1: 3 nodes, 2 elements
  col_2: 3 nodes, 2 elements


## 5. Per-column base + tip node identification

Each column's base is the mesh node at $z = 0$; the tip is the
node at $z = L$. We classify the part's nodes by coordinate.


In [5]:
tag_to_idx = {int(t): i for i, t in enumerate(fem.nodes.ids)}

def base_and_tip(label: str) -> tuple[int, int]:
    """Return (base_node_id, tip_node_id) for part ``label``."""
    base = tip = None
    for nid in fem.nodes.get(target=label).ids:
        z = float(fem.nodes.coords[tag_to_idx[int(nid)], 2])
        if abs(z - 0.0) < 1e-9:
            base = int(nid)
        elif abs(z - L) < 1e-9:
            tip = int(nid)
    assert base is not None and tip is not None, \
        f"could not find base/tip nodes for part '{label}'"
    return base, tip

column_nodes = {lbl: base_and_tip(lbl) for lbl in sorted(g.parts.labels())}
for lbl, (b, t) in column_nodes.items():
    print(f"  {lbl}: base={b}, tip={t}")


  col_0: base=1, tip=2
  col_1: base=3, tip=4
  col_2: base=5, tip=6


## 6. OpenSees ingest + analysis

Standard pattern. Element emission iterates per part label via
``fem.elements.get(target=lbl)`` — now that slot 10's FEMData gap
is fixed, the part label is a first-class target.


In [6]:
ops.wipe()
ops.model("basic", "-ndm", 3, "-ndf", 6)

for nid, xyz in fem.nodes.get():
    ops.node(int(nid), float(xyz[0]), float(xyz[1]), float(xyz[2]))

ops.geomTransf("Linear", 1, 1.0, 0.0, 0.0)      # vecxz = +x; columns run along +z

for lbl in sorted(g.parts.labels()):
    for group in fem.elements.get(target=lbl):
        for eid, nodes in zip(group.ids, group.connectivity):
            ops.element("elasticBeamColumn", int(eid),
                        int(nodes[0]), int(nodes[1]),
                        A, E, G, J, Iy, Iz, 1)

# Fix each base; apply horizontal load at each tip
for lbl, (base, tip) in column_nodes.items():
    ops.fix(int(base), 1, 1, 1, 1, 1, 1)

ops.timeSeries("Constant", 1)
ops.pattern("Plain", 1, 1)
for lbl, (base, tip) in column_nodes.items():
    ops.load(int(tip), P, 0.0, 0.0, 0.0, 0.0, 0.0)     # +x tip load

ops.system("BandGeneral"); ops.numberer("Plain"); ops.constraints("Plain")
ops.test("NormUnbalance", 1e-10, 10); ops.algorithm("Linear")
ops.integrator("LoadControl", 1.0); ops.analysis("Static")
ops.analyze(1)
print("analysis converged")


analysis converged


## 7. Verification — all three columns must give identical δ_tip

Each vertical cantilever loaded horizontally at its tip has the
classical $\delta_{\text{tip}} = P L^{3} / (3 E I)$. With
vecxz = ($+$x, 0, 0) and the column along $+z$, in-plane bending
is about local $+y$ — controlled by $I_{y}$.


In [7]:
analytical = P * L**3 / (3.0 * E * Iy)
print(f"{'part':>6s}  {'tip dx':>14s}  {'error %':>10s}")
worst = 0.0
for lbl, (base, tip) in column_nodes.items():
    d_tip = ops.nodeDisp(int(tip), 1)        # u_x (horizontal)
    err = abs(d_tip - analytical) / abs(analytical) * 100.0
    worst = max(worst, err)
    print(f"{lbl:>6s}  {d_tip:>14.6e}  {err:>9.4f} %")

print(f"\nanalytical reference : {analytical:.6e}  m")
print(f"worst-case error     : {worst:.4f} %")


  part          tip dx     error %
 col_0    4.285714e-02     0.0000 %
 col_1    4.285714e-02     0.0000 %
 col_2    4.285714e-02     0.0000 %

analytical reference : 4.285714e-02  m
worst-case error     : 0.0000 %


## What this unlocks

* **Template-plus-transforms assembly workflow** via the canonical
  ``g.parts.add(part, translate=..., rotate=...)``. Build the
  geometry once as a ``Part``, then place it many times with
  translate/rotate. Works for CAD imports via
  ``g.parts.import_step(file_path, translate=..., rotate=...)``.
* **Per-part node + element queries** via
  ``fem.nodes.get(target=part_label)`` and
  ``fem.elements.get(target=part_label)``. The label resolves
  through the same chain used everywhere else in apeGmsh:
  label → PG → part.
* **Coordinate-based role extraction** for post-mesh part
  navigation. ``base_and_tip(label)`` classifies the part's
  mesh nodes by coordinate — the pattern scales to any
  assembly geometry where specific nodes need to be addressed.


In [8]:
g_ctx.__exit__(None, None, None)


## 9. Capture results — manual + DomainCapture paths

Two ways to produce a native-HDF5 results file consumable by the
apeGmsh ``ResultsViewer``:

1. **Manual path** — query the live OpenSees domain post-analysis,
   open a ``NativeWriter``, and write nodal displacements yourself.
   Good for one-shot snapshots and post-hoc diagnostics.
2. **DomainCapture path** — declare what to capture with
   ``Recorders().nodes(...)``, hand the spec to a ``DomainCapture``
   context, and call ``cap.step(t=...)`` after each ``ops.analyze``
   (the helper does it for you). Scales to multi-stage, transient,
   modal, and multi-recorder runs.

Both produce a file that ``Results.from_native(path).viewer()`` can
open. The viewer launch is gated on ``APEGMSH_SKIP_VIEWER`` so this
notebook is safe to run under nbconvert / CI.


In [9]:
# --- EOS-WIRING-V1 ---
# Manual path: pull displacements off the live domain, write h5 yourself.
from pathlib import Path
import numpy as np
from apeGmsh.results.writers import NativeWriter

results_manual = Path("10b_part_assembly_manual.h5")
if results_manual.exists():
    results_manual.unlink()

_n = len(fem.nodes.ids)
_ux = np.array([ops.nodeDisp(int(nid), 1) for nid in fem.nodes.ids])
_uy = np.array([ops.nodeDisp(int(nid), 2) for nid in fem.nodes.ids])
_uz = np.array([ops.nodeDisp(int(nid), 3) for nid in fem.nodes.ids])
_components = {
    "displacement_x": _ux.reshape(1, _n),
    "displacement_y": _uy.reshape(1, _n),
    "displacement_z": _uz.reshape(1, _n),
}

with NativeWriter(results_manual) as _nw:
    _nw.open(fem=fem)
    _sid = _nw.begin_stage(name="static", kind="static", time=np.array([1.0]))
    _nw.write_nodes(
        _sid, "partition_0",
        node_ids=np.asarray(fem.nodes.ids, dtype=np.int64),
        components=_components,
    )
    _nw.end_stage()

print(f"manual -> {results_manual} ({results_manual.stat().st_size/1024:.1f} KB)")


manual -> 10b_part_assembly_manual.h5 (36.8 KB)

In [10]:
# DomainCapture path: declarative recorders, capture during analyze.
from apeGmsh.solvers.Recorders import Recorders
from apeGmsh.results.capture._domain import DomainCapture

recs = Recorders()
recs.nodes(components="displacement")
recs.nodes(components="reaction_force")
spec = recs.resolve(fem, ndm=3, ndf=6)

results_capture = Path("10b_part_assembly_capture.h5")
if results_capture.exists():
    results_capture.unlink()

with DomainCapture(spec, results_capture, fem, ndm=3, ndf=6) as cap:
    cap.begin_stage("run", kind="static")
    # --- copied from cell 12 ---
    ops.wipe()
    ops.model("basic", "-ndm", 3, "-ndf", 6)

    for nid, xyz in fem.nodes.get():
        ops.node(int(nid), float(xyz[0]), float(xyz[1]), float(xyz[2]))

    ops.geomTransf("Linear", 1, 1.0, 0.0, 0.0)      # vecxz = +x; columns run along +z

    for lbl in sorted(g.parts.labels()):
        for group in fem.elements.get(target=lbl):
            for eid, nodes in zip(group.ids, group.connectivity):
                ops.element("elasticBeamColumn", int(eid),
                            int(nodes[0]), int(nodes[1]),
                            A, E, G, J, Iy, Iz, 1)

    # Fix each base; apply horizontal load at each tip
    for lbl, (base, tip) in column_nodes.items():
        ops.fix(int(base), 1, 1, 1, 1, 1, 1)

    ops.timeSeries("Constant", 1)
    ops.pattern("Plain", 1, 1)
    for lbl, (base, tip) in column_nodes.items():
        ops.load(int(tip), P, 0.0, 0.0, 0.0, 0.0, 0.0)     # +x tip load

    ops.system("BandGeneral"); ops.numberer("Plain"); ops.constraints("Plain")
    ops.test("NormUnbalance", 1e-10, 10); ops.algorithm("Linear")
    ops.integrator("LoadControl", 1.0); ops.analysis("Static")
    ops.analyze(1)
    cap.step(t=ops.getTime())
    print("analysis converged")
    cap.end_stage()

print(f"capture -> {results_capture} ({results_capture.stat().st_size/1024:.1f} KB)")


analysis converged


capture -> 10b_part_assembly_capture.h5 (37.7 KB)


In [11]:
# Open the captured run in the apeGmsh ResultsViewer (subprocess,
# non-blocking). Set APEGMSH_SKIP_VIEWER=1 to skip in headless / CI.
import os
from apeGmsh.results import Results
results = Results.from_native(results_capture)
if os.environ.get("APEGMSH_SKIP_VIEWER"):
    print("[skip viewer] APEGMSH_SKIP_VIEWER set")
else:
    handle = results.viewer(blocking=False)
    print(f"viewer pid: {handle.pid}  -- close window to exit.")


[skip viewer] APEGMSH_SKIP_VIEWER set
